In [26]:
# Következő meccsek adatai

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from nba_api.stats.endpoints import scoreboardv2
import nba_api_module as nbam
import time
import json

team_id_dict = nbam.TEAM_IDS
team_abr_dict = nbam.TEAM_ABBREVIATIONS

def get_upcoming_games(days_ahead=3):
    """
    Lekéri a következő N napra tervezett meccseket.
    Maximum 1 meccs csapatonként.
    """
    upcoming = []
    teams_seen = set()
    
    today = datetime.now()
    
    for day_offset in range(days_ahead):
        check_date = today + timedelta(days=day_offset)
        date_str = check_date.strftime('%Y-%m-%d')
        
        print(f"\nEllenőrzés: {date_str}")
        
        try:
            scoreboard = scoreboardv2.ScoreboardV2(game_date=date_str)
            games = scoreboard.get_data_frames()[0]  # GameHeader
            
            if len(games) == 0:
                print(f"  Nincs meccs ezen a napon")
                continue
            
            for _, game in games.iterrows():
                game_id = str(game['GAME_ID'])
                home_team_id = game['HOME_TEAM_ID']
                home_team = team_id_dict[home_team_id]
                home_team_abr = team_abr_dict[home_team]
                away_team_id = game['VISITOR_TEAM_ID']
                away_team = team_id_dict[away_team_id]
                away_team_abr = team_abr_dict[away_team]
                
                # Csak akkor adjuk hozzá, ha egyik csapat sem szerepelt még
                if home_team_id not in teams_seen and away_team_id not in teams_seen:
                    upcoming.append({
                        'game_id': game_id,
                        'game_date': date_str,
                        'home_team_id': home_team_id,
                        'away_team_id': away_team_id,
                        'home_team': home_team,
                        'away_team': away_team
                    })
                    teams_seen.add(home_team_id)
                    teams_seen.add(away_team_id)
                    print(f"  ✓ {home_team_abr} vs. {away_team_abr} (ID: {game_id})")
            
            time.sleep(1)  # Rate limit
            
        except Exception as e:
            print(f"  Hiba {date_str} lekérésekor: {e}")
            continue
    
    print(f"\n{'='*50}")
    print(f"Összesen {len(upcoming)} meccs találva")
    print(f"{'='*50}")
    
    return pd.DataFrame(upcoming)


def create_pregame_features(game_id, season, game_date, team_ids):
    """
    Egy adott meccsre létrehozza az összes pregame feature-t.
    """
    print(f"\n[{game_id}] Pregame features létrehozása...")
    
    features = {}
    
    # 1) Pregame stats
    try:
        pregame = nbam.extract_pregame(game_id, season, game_date, team_ids)
        features.update(pregame)
        print(f"  ✓ Pregame stats")
    except Exception as e:
        print(f"  ✗ Pregame stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 2) Injury data
    try:
        injury = nbam.extract_injury(None)
        # Eltávolítjuk a missing_starters feature-t (mint az ml.ipynb-ben)
        injury.pop('home_missing_starters', None)
        injury.pop('away_missing_starters', None)
        features.update(injury)
        print(f"  ✓ Injury data")
    except Exception as e:
        print(f"  ✗ Injury data hiba: {e}")
        # Ha nincs injury adat, nullázzuk
        features['home_injury_count'] = 0
        features['away_injury_count'] = 0
    
    time.sleep(0.01)
    
    # 3) Advanced stats
    try:
        advanced = nbam.extract_advanced_stats(game_id, game_date, season, home_id=team_ids[0], away_id=team_ids[1])
        features.update(advanced)
        print(f"  ✓ Advanced stats")
    except Exception as e:
        print(f"  ✗ Advanced stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 4) Form metrics
    try:
        form = nbam.extract_form(game_id, season, game_date, home_id=team_ids[0], away_id=team_ids[1])
        features.update(form)
        print(f"  ✓ Form metrics")
    except Exception as e:
        print(f"  ✗ Form metrics hiba: {e}")
        return None
    
    return features


def calculate_differential_features(features_dict):
    """
    Differenciális feature-ök hozzáadása (mint az ml.ipynb feature engineering cellájában)
    """
    df = pd.DataFrame([features_dict])
    
    # Offensive/Defensive Rating különbségek
    df['ORtg_diff'] = df['home_ORtg'] - df['away_ORtg']
    df['DRtg_diff'] = df['home_DRtg'] - df['away_DRtg']
    df['NET_rtg_diff'] = df['home_NET_rtg'] - df['away_NET_rtg']
    
    # Pace különbség
    df['PACE_diff'] = df['home_PACE'] - df['away_PACE']
    
    # Hatékonyság különbségek
    df['TS_diff'] = df['home_TS%'] - df['away_TS%']
    df['EFG_diff'] = df['home_EFG%'] - df['away_EFG%']
    df['AST_ratio_diff'] = df['home_AST_ratio'] - df['away_AST_ratio']
    df['OREB_diff'] = df['home_OREB%'] - df['away_OREB%']
    df['turnover_diff'] = df['home_turnover_ratio'] - df['away_turnover_ratio']
    
    # Játékos minőség különbségek
    df['starter_PER_diff'] = df['home_starter_avg_PER'] - df['away_starter_avg_PER']
    df['bench_PER_diff'] = df['home_bench_avg_PER'] - df['away_bench_avg_PER']
    df['star_usage_diff'] = df['home_star_usage'] - df['away_star_usage']
    df['avg_TS_diff'] = df['home_avg_TS'] - df['away_avg_TS']
    df['top3_points_diff'] = df['home_top3_points_avg'] - df['away_top3_points_avg']
    
    # Pihenés és forma különbségek
    df['rest_days_diff'] = df['home_rest_days'] - df['away_rest_days']
    df['recent_form10_diff'] = df['home_recent_form10'] - df['away_recent_form10']
    df['recent_form5_diff'] = df['home_recent_form5'] - df['away_recent_form5']
    df['recent_form3_diff'] = df['home_recent_form3'] - df['away_recent_form3']
    
    # Sérülés különbség
    df['injury_count_diff'] = df['home_injury_count'] - df['away_injury_count']
    
    # Back-to-back advantage
    df['b2b_advantage'] = df['away_is_back_to_back'].astype(int) - df['home_is_back_to_back'].astype(int)
    
    return df.iloc[0].to_dict()


def align_features_to_model(features_dict, feature_columns_path='models/feature_columns.json'):
    """
    Biztosítja, hogy a feature-ök pontosan egyezzenek a modell által elvárt feature listával.
    """
    with open(feature_columns_path, 'r') as f:
        expected_features = json.load(f)
    
    # Ellenőrizzük, hogy minden szükséges feature megvan-e
    missing_features = [f for f in expected_features if f not in features_dict]
    if missing_features:
        print(f"\n⚠️  Hiányzó features: {missing_features}")
        # Nullával töltjük fel a hiányzó feature-öket
        for feat in missing_features:
            features_dict[feat] = 0
    
    # Csak a modell által elvárt feature-öket tartjuk meg, megfelelő sorrendben
    aligned_features = {feat: features_dict[feat] for feat in expected_features}
    
    return pd.DataFrame([aligned_features])


def prepare_upcoming_games_for_prediction(days_ahead=3, output_file='data/upcoming_games_features.csv'):
    """
    Teljes pipeline: lekéri a következő meccseket és előkészíti a predikcióhoz.
    """
    print("="*60)
    print("KÖVETKEZŐ MECCSEK ELŐKÉSZÍTÉSE PREDIKCIÓHOZ")
    print("="*60)
    
    # 1) Következő meccsek lekérése
    upcoming_df = get_upcoming_games(days_ahead=days_ahead)
    
    if len(upcoming_df) == 0:
        print("\n❌ Nem találtunk következő meccseket!")
        return None
    
    # 2) Features gyűjtése minden meccshez
    all_features = []
    
    for idx, game in upcoming_df.iterrows():
        game_id = game['game_id']
        game_date = game['game_date']
        
        print(f"\n{'='*60}")
        print(f"[{idx+1}/{len(upcoming_df)}] {game['away_team']} @ {game['home_team']}")
        print(f"{'='*60}")
        
        try:
            # Pregame features
            features = create_pregame_features(game_id, season='2025-26', game_date=game_date, team_ids=[game['home_team_id'], game['away_team_id']])
            
            if features is None:
                print(f"  ⚠️  Kihagyva (hiányos adatok)")
                continue
            
            # Differenciális features
            features = calculate_differential_features(features)
            
            # Game meta info hozzáadása
            features['game_id'] = game_id
            features['game_date'] = game_date
            features['home_team'] = game['home_team']
            features['away_team'] = game['away_team']
            
            all_features.append(features)
            
            print(f"  ✅ Sikeres feldolgozás")
            
        except Exception as e:
            print(f"  ❌ Hiba: {e}")
            continue
        
        time.sleep(2)  # Rate limit
    
    if len(all_features) == 0:
        print("\n❌ Egyetlen meccshez sem sikerült adatot gyűjteni!")
        return None
    
    # 3) DataFrame összeállítása
    features_df = pd.DataFrame(all_features)
    
    # 4) Feature alignment a modell által elvárt formátumra
    print(f"\n{'='*60}")
    print("FEATURE ALIGNMENT")
    print(f"{'='*60}")
    
    # Meta információkat külön tároljuk
    meta_cols = ['game_id', 'game_date', 'home_team', 'away_team']
    meta_df = features_df[meta_cols].copy()
    
    # Feature-ök alignment
    aligned_features = []
    for idx, row in features_df.iterrows():
        row_dict = row.to_dict()
        aligned = align_features_to_model(row_dict)
        aligned_features.append(aligned)
    
    aligned_df = pd.concat(aligned_features, ignore_index=True)
    
    # Meta info visszacsatolása
    final_df = pd.concat([meta_df.reset_index(drop=True), aligned_df], axis=1)
    
    # 5) Mentés
    #final_df.to_csv(output_file, index=False)
    
    print(f"\n{'='*60}")
    print("✅ KÉSZ!")
    print(f"{'='*60}")
    print(f"Meccsek száma: {len(final_df)}")
    print(f"Feature-ök száma: {len(aligned_df.columns)}")
    print(f"Mentve: {output_file}")
    print(f"\nMeccsek:")
    for _, game in final_df.iterrows():
        print(f"  • {game['away_team']} @ {game['home_team']} ({game['game_date']})")
    
    return final_df


# HASZNÁLAT:
upcoming_features = prepare_upcoming_games_for_prediction(days_ahead=3)

upcoming_features

KÖVETKEZŐ MECCSEK ELŐKÉSZÍTÉSE PREDIKCIÓHOZ

Ellenőrzés: 2025-11-21
  ✓ CLE vs. IND (ID: 0022500048)
  ✓ BOS vs. BKN (ID: 0022500049)
  ✓ TOR vs. WAS (ID: 0022500050)
  ✓ CHI vs. MIA (ID: 0022500051)
  ✓ DAL vs. NOP (ID: 0022500052)
  ✓ PHX vs. MIN (ID: 0022500053)
  ✓ HOU vs. DEN (ID: 0022500054)
  ✓ UTA vs. OKC (ID: 0022500055)
  ✓ GSW vs. POR (ID: 0022500056)

Ellenőrzés: 2025-11-22
  ✓ CHA vs. LAC (ID: 0022500268)
  ✓ ORL vs. NYK (ID: 0022500269)
  ✓ MIL vs. DET (ID: 0022500272)

Ellenőrzés: 2025-11-23

Összesen 12 meccs találva

[1/12] Indiana Pacers @ Cleveland Cavaliers

[0022500048] Pregame features létrehozása...
  ✓ Pregame stats
  ✓ Injury data
  ✓ Advanced stats
  ✓ Form metrics
  ✅ Sikeres feldolgozás

[2/12] Brooklyn Nets @ Boston Celtics

[0022500049] Pregame features létrehozása...
  ✓ Pregame stats
  ✓ Injury data
  ✓ Advanced stats
  ✓ Form metrics
  ✅ Sikeres feldolgozás

[3/12] Washington Wizards @ Toronto Raptors

[0022500050] Pregame features létrehozása...
  ✓ Pr

,game_id,game_date,home_team,away_team,home_ORtg,away_ORtg,home_DRtg,away_DRtg,home_NET_rtg,away_NET_rtg,...,bench_PER_diff,star_usage_diff,avg_TS_diff,top3_points_diff,rest_days_diff,recent_form10_diff,recent_form5_diff,recent_form3_diff,injury_count_diff,b2b_advantage
0,0022500048,2025-11-21,Cleveland Cavaliers,Indiana Pacers,91.896528,81.980636,88.973854,91.607902,2.922674,-9.627266,...,0.180852,0.275287,0.022782,2.725620,0,0.5,0.4,0.333333,0,0
1,0022500049,2025-11-21,Boston Celtics,Brooklyn Nets,89.953283,88.533856,85.840531,97.935171,4.112753,-9.401315,...,1.820734,0.286417,0.074612,8.857222,0,0.4,0.6,0.666667,0,0
2,0022500050,2025-11-21,Toronto Raptors,Washington Wizards,97.395035,88.283043,93.386563,101.046857,4.008472,-12.763813,...,0.392921,0.142738,0.046575,3.845464,0,0.9,1.0,1.000000,0,0
3,0022500051,2025-11-21,Chicago Bulls,Miami Heat,95.357479,96.508214,95.469401,93.541133,-0.111922,2.967081,...,0.245638,0.111936,0.020199,-2.156117,0,-0.2,-0.2,0.000000,0,0
4,0022500052,2025-11-21,Dallas Mavericks,New Orleans Pelicans,86.845705,84.429173,92.061368,94.608210,-5.215663,-10.179037,...,0.199834,0.076505,0.044921,-1.178463,0,0.0,0.2,0.333333,0,0
5,0022500053,2025-11-21,Phoenix Suns,Minnesota Timberwolves,92.256153,96.754525,88.184512,91.512986,4.071640,5.241539,...,0.633119,-0.245421,-0.023391,2.924918,1,0.0,0.0,0.000000,0,0
6,0022500054,2025-11-21,Houston Rockets,Denver Nuggets,91.372631,99.237944,82.978373,90.025023,8.394258,9.212921,...,-0.356033,-0.224815,-0.070962,-5.010031,0,0.1,0.2,0.333333,0,0
7,0022500055,2025-11-21,Utah Jazz,Oklahoma City Thunder,88.603906,97.315184,92.631356,84.981575,-4.027450,12.333609,...,-0.015999,-0.235907,-0.051467,-4.046045,1,-0.6,-0.6,-0.666667,0,0
8,0022500056,2025-11-21,Golden State Warriors,Portland Trail Blazers,91.392895,86.918126,91.251857,88.165984,0.141038,-1.247858,...,0.530080,0.149789,-0.001280,-0.512428,0,0.2,0.4,0.333333,0,0
9,0022500268,2025-11-22,Charlotte Hornets,LA Clippers,89.785316,92.344553,92.557418,97.447075,-2.772102,-5.102522,...,-0.604339,-0.328219,-0.049186,-0.190935,1,0.1,0.0,0.000000,0,0


In [27]:
# Predikciók készítése az összes modellel

import pandas as pd
import numpy as np
import joblib
import json

# -------------------------------------------------------------------------
# 1) ADATOK ÉS MODELLEK BETÖLTÉSE
# -------------------------------------------------------------------------

# Feature-ök betöltése
df = upcoming_features.copy()

# Meta oszlopok elkülönítése
meta_cols = ['game_id', 'game_date', 'home_team', 'away_team']
meta_df = df[meta_cols].copy()

# Feature columns betöltése
with open('models/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

X = df[feature_cols]

# Közös scaler betöltése
scaler = joblib.load('models/scaler.joblib')
X_scaled = scaler.transform(X)

print(f"Meccsek száma: {len(X)}")
print(f"Feature-ök száma: {len(feature_cols)}")

# -------------------------------------------------------------------------
# 2) PREDIKCIÓK - MINDEN MODELLRE
# -------------------------------------------------------------------------

predictions = meta_df.copy()

# --- Random Forest + PCA ---
pca_rf = joblib.load('models/rf_pca_transformer.joblib')
model_rf = joblib.load('models/rf_pca_model.joblib')

X_pca_rf = pca_rf.transform(X_scaled)
predictions['rf_pca_prob_home'] = model_rf.predict_proba(X_pca_rf)[:, 1]

print(f"\n✓ Random Forest + PCA ({pca_rf.n_components_} komponens)")

# --- XGBoost Very Shallow + PCA ---
pca_xgb_shallow = joblib.load('models/xgb_shallow_pca_transformer.joblib')
model_xgb_shallow = joblib.load('models/xgb_shallow_model.joblib')

X_pca_xgb_shallow = pca_xgb_shallow.transform(X_scaled)
predictions['xgb_shallow_prob_home'] = model_xgb_shallow.predict_proba(X_pca_xgb_shallow)[:, 1]

print(f"✓ XGBoost Shallow + PCA ({pca_xgb_shallow.n_components_} komponens)")

# --- XGBoost Tuned + PCA ---
pca_xgb_tuned = joblib.load('models/xgb_tuned_pca_transformer.joblib')
model_xgb_tuned = joblib.load('models/xgb_tuned_model.joblib')

X_pca_xgb_tuned = pca_xgb_tuned.transform(X_scaled)
predictions['xgb_tuned_prob_home'] = model_xgb_tuned.predict_proba(X_pca_xgb_tuned)[:, 1]

print(f"✓ XGBoost Tuned + PCA ({pca_xgb_tuned.n_components_} komponens)")

# -------------------------------------------------------------------------
# 3) ENSEMBLE ÉS ÖSSZEGZÉS
# -------------------------------------------------------------------------

# Ensemble átlag
predictions['ensemble_prob_home'] = predictions[['rf_pca_prob_home', 'xgb_shallow_prob_home', 'xgb_tuned_prob_home']].mean(axis=1)

# Away prob számítás
for col in ['rf_pca', 'xgb_shallow', 'xgb_tuned', 'ensemble']:
    predictions[f'{col}_prob_away'] = 1 - predictions[f'{col}_prob_home']

# -------------------------------------------------------------------------
# 4) EREDMÉNYEK MEGJELENÍTÉSE
# -------------------------------------------------------------------------

print(f"\n{'='*80}")
print("PREDIKCIÓK")
print(f"{'='*80}\n")

display_cols = ['home_team', 'away_team', 'rf_pca_prob_home', 'xgb_shallow_prob_home', 
                'xgb_tuned_prob_home', 'ensemble_prob_home']

print(predictions[display_cols].to_string(index=False))

# Mentés
#predictions.to_csv('data/predictions_upcoming.csv', index=False)
#print(f"\n✅ Mentve: data/predictions_upcoming.csv")

Meccsek száma: 12
Feature-ök száma: 60

✓ Random Forest + PCA (17 komponens)
✓ XGBoost Shallow + PCA (10 komponens)
✓ XGBoost Tuned + PCA (17 komponens)

PREDIKCIÓK

            home_team              away_team  rf_pca_prob_home  xgb_shallow_prob_home  xgb_tuned_prob_home  ensemble_prob_home
  Cleveland Cavaliers         Indiana Pacers          0.830193               0.734006             0.919009            0.827736
       Boston Celtics          Brooklyn Nets          0.714712               0.743474             0.900234            0.786140
      Toronto Raptors     Washington Wizards          0.696627               0.737161             0.826356            0.753381
        Chicago Bulls             Miami Heat          0.388700               0.408594             0.407205            0.401500
     Dallas Mavericks   New Orleans Pelicans          0.745360               0.681107             0.821894            0.749453
         Phoenix Suns Minnesota Timberwolves          0.643352          

In [31]:
# Oddsok

from pathlib import Path
import os
import sys

teams_tx_map = {
    "Atlanta Hawks":"Atlanta",
    "Miami Heat":"Miami",
    "Orlando Magic":"Orlando",
    "New York Knicks":"New York",
    "Milwaukee Bucks":"Milwaukee",
    "Phoenix Suns":"Phoenix",
    "Dallas Mavericks":"Dallas",
    "Minnesota Timberwolves":"Minnesota",
    "Toronto Raptors":"Toronto",
    "Cleveland Cavaliers":"Cleveland",
    "Brooklyn Nets":"Brooklyn",
    "Los Angeles Lakers":"LA Lakers",
    "Houston Rockets":"Houston",
    "New Orleans Pelicans":"New Orleans",
    "Golden State Warriors":"Golden State",
    "Boston Celtics":"Boston",
    "Denver Nuggets":"Denver",
    "San Antonio Spurs":"San Antonio",
    "LA Clippers":"LA Clippers",
    "Chicago Bulls":"Chicago",
    "Indiana Pacers":"Indiana",
    "Portland Trail Blazers":"Portland",
    "Philadelphia 76ers":"Philadelphia",
    "Oklahoma City Thunder":"Oklahoma City",
    "Utah Jazz":"Utah",
    "Sacramento Kings":"Sacramento",
    "Detroit Pistons":"Detroit",
    "Memphis Grizzlies":"Memphis",
    "Washington Wizards":"Washington"
}

teams_tx_map_rev = {v:k for k,v in teams_tx_map.items()}

nb_dir = Path(os.getcwd())
root = nb_dir.parents[2]
sys.path.append(str(root / "modules"))

from tx_module import get_league_odds

url = "https://www.tippmixpro.hu/hu/fogadas/i/bajnoksag-lokacio/kosarlabda/8/usa/229/nba/274663790763708416"
odds = get_league_odds(url)

odds['home_team_api'] = odds['home_team'].map(teams_tx_map_rev)
odds['away_team_api'] = odds['away_team'].map(teams_tx_map_rev)

preds_w_odds = pd.merge(predictions, 
                        odds[['home_team_api', 'away_team_api','home_odds', 'away_odds']],
                        left_on=['home_team', 'away_team'],
                        right_on=['home_team_api', 'away_team_api'],
                        how="inner")

preds_w_odds.drop(columns=['home_team_api', 'away_team_api'], inplace=True)
display(preds_w_odds)

,game_id,game_date,home_team,away_team,rf_pca_prob_home,xgb_shallow_prob_home,xgb_tuned_prob_home,ensemble_prob_home,rf_pca_prob_away,xgb_shallow_prob_away,xgb_tuned_prob_away,ensemble_prob_away,home_odds,away_odds
0,0022500048,2025-11-21,Cleveland Cavaliers,Indiana Pacers,0.830193,0.734006,0.919009,0.827736,0.169807,0.265994,0.080991,0.172264,1.12,7.00
1,0022500049,2025-11-21,Boston Celtics,Brooklyn Nets,0.714712,0.743474,0.900234,0.786140,0.285288,0.256526,0.099766,0.213860,1.11,7.50
2,0022500050,2025-11-21,Toronto Raptors,Washington Wizards,0.696627,0.737161,0.826356,0.753381,0.303373,0.262839,0.173644,0.246619,1.12,7.00
3,0022500051,2025-11-21,Chicago Bulls,Miami Heat,0.388700,0.408594,0.407205,0.401500,0.611300,0.591406,0.592795,0.598500,1.74,2.20
4,0022500052,2025-11-21,Dallas Mavericks,New Orleans Pelicans,0.745360,0.681107,0.821894,0.749453,0.254640,0.318893,0.178106,0.250547,1.58,2.51
5,0022500053,2025-11-21,Phoenix Suns,Minnesota Timberwolves,0.643352,0.534350,0.676312,0.618005,0.356648,0.465650,0.323688,0.381995,2.54,1.57
6,0022500054,2025-11-21,Houston Rockets,Denver Nuggets,0.696530,0.541484,0.918467,0.718827,0.303470,0.458516,0.081533,0.281173,1.74,2.20
7,0022500055,2025-11-21,Utah Jazz,Oklahoma City Thunder,0.280399,0.334099,0.283333,0.299277,0.719601,0.665901,0.716667,0.700723,8.75,1.09
8,0022500056,2025-11-21,Golden State Warriors,Portland Trail Blazers,0.530238,0.619335,0.283294,0.477622,0.469762,0.380665,0.716706,0.522378,1.33,3.55
